In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("CrisisMMD")
IMAGES = ROOT / "CrisisMMD_v2.0"
SPLITS = ROOT / "crisismmd_datasplit_all"

train_tsv = SPLITS / "task_informative_text_img_train.tsv"
dev_tsv = SPLITS / "task_informative_text_img_dev.tsv"
test_tsv = SPLITS / "task_informative_text_img_test.tsv"

def load_split(data_path):

    features = pd.read_csv(data_path, sep="\t")

    # label_image for images which is then converted to binary
    features["informative"] = features["label_image"].map({"informative": 1, "not_informative": 0})

    features["img_path"] = features["image"].apply(lambda p: str((IMAGES / p).resolve()))

    features = features.dropna(subset=["informative", "img_path"])
    return features[["tweet_id", "image_id", "img_path", "informative"]]

train = load_split(train_tsv)
dev = load_split(dev_tsv)
test = load_split(test_tsv)

# #Sanity Checks for Data Loading
# print(train.head())
# print(train["informative"].value_counts())

# missing = train[~train["img_path"].apply(lambda p: Path(p).exists())]
# print("Missing images:", len(missing))
# print(missing.head(3))

             tweet_id              image_id  \
0  917791291823591425  917791291823591425_0   
1  917791291823591425  917791291823591425_1   
2  917793137925459968  917793137925459968_0   
3  917793137925459968  917793137925459968_1   
4  917793137925459968  917793137925459968_2   

                                            img_path  informative  
0  F:\CompSci\CompSci Repo\ITEC5920\ITEC5920\Proj...            1  
1  F:\CompSci\CompSci Repo\ITEC5920\ITEC5920\Proj...            0  
2  F:\CompSci\CompSci Repo\ITEC5920\ITEC5920\Proj...            1  
3  F:\CompSci\CompSci Repo\ITEC5920\ITEC5920\Proj...            1  
4  F:\CompSci\CompSci Repo\ITEC5920\ITEC5920\Proj...            1  
informative
1    7059
0    6549
Name: count, dtype: int64
Missing images: 0
Empty DataFrame
Columns: [tweet_id, image_id, img_path, informative]
Index: []


In [7]:
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def load_image(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    label = tf.cast(label, tf.float32)
    return img, label

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["img_path"].values, df["informative"].values))
    if training:
        ds = ds.shuffle(min(len(df), 5000), reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train, training=True)
dev_ds   = make_ds(dev, training=False)
test_ds  = make_ds(test, training=False)

for x, y in train_ds.take(1):
    print(x.shape, y.shape)

(16, 224, 224, 3) (16,)


In [9]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, Model

base = VGG16(weights="imagenet", include_top=False, input_shape=(224,224,3))
base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=dev_ds,
    epochs=6,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
    ]
)

Epoch 1/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 383s 446ms/step - accuracy: 0.7116 - auc: 0.7630 - loss: 0.7479 - val_accuracy: 0.7769 - val_auc: 0.8631 - val_loss: 0.4794
Epoch 2/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 304s 357ms/step - accuracy: 0.7646 - auc: 0.8445 - loss: 0.4905 - val_accuracy: 0.7702 - val_auc: 0.8713 - val_loss: 0.4636
Epoch 3/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 293s 344ms/step - accuracy: 0.7706 - auc: 0.8568 - loss: 0.4706 - val_accuracy: 0.7912 - val_auc: 0.8783 - val_loss: 0.4523
Epoch 4/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 298s 349ms/step - accuracy: 0.7842 - auc: 0.8703 - loss: 0.4514 - val_accuracy: 0.7953 - val_auc: 0.8802 - val_loss: 0.4480
Epoch 5/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 292s 342ms/step - accuracy: 0.7953 - auc: 0.8766 - loss: 0.4403 - val_accuracy: 0.8011 - val_auc: 0.8885 - val_loss: 0.4361
Epoch 6/6
851/851 ━━━━━━━━━━━━━━━━━━━━ 291s 341ms/step - accuracy: 0.7934 - auc: 0.8795 - loss: 0.4363 - val_accuracy: 0.7979 - val_auc: 0.8874 - val_loss: 0.4336
